In [1]:
import os

os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=12")
import jax

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
print("JAX devices:", jax.devices())

JAX devices: [CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7), CpuDevice(id=8), CpuDevice(id=9), CpuDevice(id=10), CpuDevice(id=11)]


In [2]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
for parent in [repo_root, *repo_root.parents]:
    src = parent / "src"
    if (src / "gwemfish").is_dir():
        sys.path.insert(0, str(src))
        break

import matplotlib.pyplot as plt

from gwemfish.simple_pipeline import (
    _deep_merge_dict,
    make_default_cfg,
    plot_system_observation,
    setup_em_observation,
    setup_gw_observation,
    run_inference, 
    plot_posterior, 
    to_source_plane_samples, 
    plot_source_posterior,
)
#
#OUTPUT_DIR = os.path.join("examples", "outputs", "simple_pipeline_demonstration")
#os.makedirs(OUTPUT_DIR, exist_ok=True)
#print("OUTPUT_DIR:", os.path.abspath(OUTPUT_DIR))

In [3]:
from gwemfish.config import DEFAULT_KWARGS_NUMERICS, SOLVER_PARAMS
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
import numpy as np
import matplotlib.pyplot as plt

In [5]:
sample_cfg = {
    "em": {
        "pixel_grid_kwargs": {"npix": int(40), "pix_scl": 0.1},
        "psf_kwargs": {"psf_type": "GAUSSIAN", "fwhm": 0.16, "pixel_size": 0.1},#0.067
        "noise_simu_kwargs": {"npix": int(40), "background_rms": 1e-2, "exposure_time": 2200},
        "noise_inf_kwargs": {"npix": int(40), "background_rms": None, "exposure_time": 2200},
        "kwargs_numerics": DEFAULT_KWARGS_NUMERICS,
        "exposure_time": 2200,
        "seed": 87651,
        "source_pos": (0.0, 0.0), #Is this the position of the source of the GW or the position of the center of the source galaxy?
        "kwargs_source": [
            {
                "amp": 250,
                "R_sersic": 0.4,
                "n_sersic": 4.0,
                "e1": 0,
                "e2": 0,
                "center_x": 0.0, #same question for these center_x and y values
                "center_y": 0.0,
            }
        ],
        "kwargs_lens_light": [
            {
                "amp": 50.0,
                "R_sersic": 2.0,
                "n_sersic": 4.0,
                "e1": 0.0,
                "e2": 0.0,
                "center_x": 0.0,
                "center_y": 0.0,
            }
        ],
    },
    "lens": {
        "lens_model_list": ["EPL", "SHEAR"],
        "kwargs_lens": [
            {
                "theta_E": 1.2,
                "e1": 0.0,
                "e2": 0.1,
                "gamma": 2.0,
                "center_x": 0.0,
                "center_y": 0.0,
            },
            {"gamma1": 0.1, "gamma2": 0.0, "ra_0": 0.0, "dec_0": 0.0},
        ],
        "zl": 0.7,
        "zs": 1.5,
    },
    "gw": {
        "source_pos": (0.0, 0.0),
        "solver_params": SOLVER_PARAMS,
        "image_box_half_width": 10.6,
        "error_scales": {
            "sigma_td": 0.05,
            "sigma_dL_eff": 3.0,
            "epsilon": 0.005,
        },
    },
    "plot": {"plot_mode": "groupwise", "save_path": None, "save_tag": None, "hist_kwargs": {"density": True}},
    "source_plane": {"filter_std": None, "use_filtered": False},
    "output": {
        "output_dir": OUTPUT_DIR,
        "save_samples_path": None,
        "save_truths_path": None,
        "save_source_samples_path": None,
        "save_system_plot_path": None,
        "json_path": None,
    },
}

In [6]:
from lenstronomy.Util import param_util
from lenstronomy.SimulationAPI.mag_amp_conversion import MagAmpConversion

In [7]:
import copy

def row_to_cfg(row, sample_cfg, gw_enabled):
    """
    Convert one row of the Qiuhan's sky catalog into a GWEMFISH configuration.

    Parameters
    ----------
    row : pandas.Series
        One row of the lens catalog.

    sample_cfg : dict
        Reference GWEMFISH config.

    Returns
    -------
    cfg : dict
        GWEMFISH configuration for this lens system.
    """

    cfg = copy.deepcopy(sample_cfg)

    # ==========================================================
    # Convert axis ratio + PA -> ellipticity
    # ==========================================================

    # --- 1. PREPARE LENS (DEFLECTOR) LIGHT PARAMETERS ---
    # Convert q and pa to lenstronomy ellipticity (e1, e2)
    lens_e1, lens_e2 = param_util.phi_q2_ellipticity(phi=row['deflector_pa'], q=row['deflector_q'])


    kwargs_lens_light_mag = [{
    'magnitude': row['deflector_app_mag_VIS'],
    'R_sersic': row['deflector_Re'],
    'n_sersic': 4.0,  # Standard assumption for elliptical lens galaxies
    'e1': lens_e1, 'e2': lens_e2,
    'center_x': 0, 'center_y': 0
    }]

    # --- 2. PREPARE SOURCE LIGHT PARAMETERS ---
    # Convert q and pa to lenstronomy ellipticity (e1, e2)
    source_e1, source_e2 = param_util.phi_q2_ellipticity(phi=row['source_pa'], q=row['source_q'])
    
    # Choose your band (e.g., VIS)
    kwargs_source_mag = [{
        'magnitude': row['source_app_mag_VIS'], 
        'R_sersic': row['source_Re'],
        'n_sersic': row['source_sersic_index'],
        'e1': source_e1, 'e2': source_e2,
        'center_x': row['source_relative_x'], 
        'center_y': row['source_relative_y']
    }]
    
    # --- 3. CONVERT BOTH TO AMP ---
    # Define your model profile types
    kwargs_model = {
        'lens_light_model_list': ['SERSIC_ELLIPSE'],
        'source_light_model_list': ['SERSIC_ELLIPSE']
    }
    
    # Initialize the converter with your survey zero-point
    mag_converter = MagAmpConversion(kwargs_model=kwargs_model, magnitude_zero_point=25.9)
    
    # Get the final dictionaries containing the calculated 'amp' keys
    lens_light_amp = mag_converter.magnitude2amplitude(kwargs_lens_light_mag=kwargs_lens_light_mag)
    source_amp = mag_converter.magnitude2amplitude(kwargs_source_mag=kwargs_source_mag)
    
    #print(lens_light_amp)
    #print(source_amp)
    
    #source_e1, source_e2 = q_pa_to_e1e2(
     #   row["source_q"],
      #  row["source_pa"],
   # )

    lens_pos = (float(row['deflector_ra']), float(row['deflector_dec']))
    
    source_pos = (float(row["source_relative_x"]), float(row["source_relative_y"]))

    # ==========================================================
    # Lens geometry
    # ==========================================================

    cfg["lens"]["zl"] = float(row["deflector_z"])
    cfg["lens"]["zs"] = float(row["source_z"])

    # ==========================================================
    # Lens mass model (EPL)
    # ==========================================================

    cfg["lens"]["kwargs_lens"][0]["theta_E"] = float(row["deflector_thetaE"])
    cfg["lens"]["kwargs_lens"][0]["gamma"] = float(row["deflector_slope"])

    cfg["lens"]["kwargs_lens"][0]["e1"] = float(lens_e1)
    cfg["lens"]["kwargs_lens"][0]["e2"] = float(lens_e2)

    cfg["lens"]["kwargs_lens"][0]["center_x"] = 0.00
    cfg["lens"]["kwargs_lens"][0]["center_y"] = 0.00

    # ==========================================================
    # External shear
    # ==========================================================

    cfg["lens"]["kwargs_lens"][1]["gamma1"] = float(row["deflector_shear1"])
    cfg["lens"]["kwargs_lens"][1]["gamma2"] = float(row["deflector_shear2"])

    cfg["lens"]["kwargs_lens"][1]["ra_0"] = 0.00
    cfg["lens"]["kwargs_lens"][1]["dec_0"] = 0.00

    # ==========================================================
    # Source light
    # ==========================================================

    cfg["em"]["kwargs_source"][0]["R_sersic"] = float(row["source_Re"])
    cfg["em"]["kwargs_source"][0]["n_sersic"] = float(row["source_sersic_index"])

    cfg["em"]["kwargs_source"][0]["e1"] = float(source_e1)
    cfg["em"]["kwargs_source"][0]["e2"] = float(source_e2)

    cfg["em"]["kwargs_source"][0]["center_x"] = source_pos[0]
    cfg["em"]["kwargs_source"][0]["center_y"] = source_pos[1]

    cfg["em"]["kwargs_source"][0]["amp"] = float(source_amp[1][0]['amp'])

    # ==========================================================
    # Lens light
    # ==========================================================

    cfg["em"]["kwargs_lens_light"][0]["R_sersic"] = float(row["deflector_Re"])
    cfg["em"]["kwargs_lens_light"][0]["n_sersic"] = float(4)

    cfg["em"]["kwargs_lens_light"][0]["e1"] = float(lens_e1)
    cfg["em"]["kwargs_lens_light"][0]["e2"] = float(lens_e2)

    cfg["em"]["kwargs_lens_light"][0]["center_x"] = 0.00
    cfg["em"]["kwargs_lens_light"][0]["center_y"] = 0.00

    cfg["em"]["kwargs_lens_light"][0]["amp"] = float(lens_light_amp[0][0]['amp'])

    # ==========================================================
    # GW source position
    # ==========================================================
    cfg["em"]["source_pos"] = source_pos
    
    if gw_enabled is True:
        cfg["gw"]["source_pos"] = (source_pos[0]+0.005, source_pos[1]-0.005)

    if gw_enabled is False:
        cfg["gw"] = {"enabled": False}

    return cfg

## Prepare to implement data from Qiuhan's catalog

In [8]:
import pandas as pd

In [9]:
candidates_ALL = pd.read_csv("candidates.csv")

In [10]:
candidates_ALL

,deflector_ra,deflector_dec,deflector_z,deflector_q,deflector_pa,deflector_vdisp,deflector_rband_abs_mag,deflector_Re,deflector_thetaE,deflector_thetaE_autolens,...,source_app_mag_r,source_app_mag_i,source_app_mag_z,IC_Euclid,IC_CSST,IC_Roman,IC_nofactor_Euclid,IC_nofactor_CSST,IC_nofactor_Roman,origin_batch
0,5.521573,12.172388,0.473488,0.946585,295.919087,278.417558,-23.967032,0.715179,1.492396,1.503382,...,25.371767,25.238202,25.109647,1282.653388,2111.513737,3556.239372,139.843750,230.211842,387.725830,R1
1,5.107674,4.398972,0.684311,0.694221,350.748688,266.539780,-23.769548,0.685350,1.099934,1.087349,...,24.868669,24.783516,24.601721,381.398181,0.000000,1731.863009,133.203125,0.000000,604.852295,R1
2,5.367231,13.574311,0.479042,0.631825,294.380575,239.691830,-23.167243,0.680128,0.819801,0.842893,...,25.392271,25.040184,24.614573,489.703625,712.600168,1250.920704,87.890625,127.895468,224.511719,R1
3,10.787265,8.119286,1.250245,0.726541,319.129660,268.454805,-23.729902,0.636792,0.734991,0.747363,...,25.253807,25.141354,25.123164,609.564924,1284.292878,1107.557697,90.234375,190.114884,163.952637,R1
4,12.851097,5.840771,0.683476,0.786874,210.701926,290.357756,-24.032233,1.052951,1.383501,1.283163,...,25.184428,25.180062,25.089573,1393.320084,2134.972594,4629.886529,180.468750,276.530741,599.682617,R1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,2.106245,12.042970,0.762288,0.573538,54.818636,289.132718,-24.073139,0.795905,0.990981,0.869206,...,25.103304,25.219699,25.099910,793.076765,1608.188856,1639.297478,133.984375,271.691453,276.947021,R20
123,9.047482,3.043769,0.379735,0.703780,240.083800,167.078085,-21.546995,0.480968,0.619740,0.518581,...,25.585207,25.607231,25.600357,271.575078,356.801559,636.763507,89.453125,117.525565,209.741211,R20
124,5.493719,13.224909,0.470081,0.956048,241.407624,188.755435,-22.060323,0.615896,0.728976,0.735072,...,25.225396,25.142130,25.036229,348.171384,659.111468,749.959646,78.515625,148.635273,169.122314,R20
125,15.131210,3.804275,0.379122,0.575450,155.060819,252.266642,-23.462695,1.168367,1.250894,1.137280,...,25.353245,24.797901,24.229633,335.064917,297.652089,2915.989626,100.390625,89.181164,873.675537,R20


In [11]:
candidates = candidates_ALL.iloc[[0,3,4,]]

In [12]:
candidates

,deflector_ra,deflector_dec,deflector_z,deflector_q,deflector_pa,deflector_vdisp,deflector_rband_abs_mag,deflector_Re,deflector_thetaE,deflector_thetaE_autolens,...,source_app_mag_r,source_app_mag_i,source_app_mag_z,IC_Euclid,IC_CSST,IC_Roman,IC_nofactor_Euclid,IC_nofactor_CSST,IC_nofactor_Roman,origin_batch
0,5.521573,12.172388,0.473488,0.946585,295.919087,278.417558,-23.967032,0.715179,1.492396,1.503382,...,25.371767,25.238202,25.109647,1282.653388,2111.513737,3556.239372,139.843750,230.211842,387.725830,R1
3,10.787265,8.119286,1.250245,0.726541,319.129660,268.454805,-23.729902,0.636792,0.734991,0.747363,...,25.253807,25.141354,25.123164,609.564924,1284.292878,1107.557697,90.234375,190.114884,163.952637,R1
4,12.851097,5.840771,0.683476,0.786874,210.701926,290.357756,-24.032233,1.052951,1.383501,1.283163,...,25.184428,25.180062,25.089573,1393.320084,2134.972594,4629.886529,180.468750,276.530741,599.682617,R1


## Defining more useful functions

In [13]:
shared_params = [
    'lens0_e1',
    'lens0_e2',
    'lens0_gamma',
    'lens0_theta_E',
    'lens1_gamma1',
    'lens1_gamma2'
]

# Fraction of lenses that the EM catalogue contains
f = 1.0

# Number of EM catalogue systems
Ncat = len(candidates)


def logmeanexp(x):
    """log(mean(exp(x))) in a numerically stable way."""
    return logsumexp(x) - np.log(len(x))


from scipy.stats import norm
def log_F_EM(samples, tp, row):
    # 1. Extract posterior sample parameters
    gamma = np.asarray(samples["lens0_gamma"])
    thetaE = np.asarray(samples["lens0_theta_E"])
    e1 = np.asarray(samples["lens0_e1"])
    e2 = np.asarray(samples["lens0_e2"])
    gamma1 = np.asarray(samples["lens1_gamma1"])
    gamma2 = np.asarray(samples["lens1_gamma2"])

    # 2. Calculate ellipticity prior widths
    phi = row["deflector_pa"]

    sigma_e = (
        2 / (1 + row["deflector_q"])**2
    ) * 0.05

    sigma_e1 = abs(np.cos(2 * phi)) * sigma_e
    sigma_e2 = abs(np.sin(2 * phi)) * sigma_e

    # 3. Evaluate log fiducial prior F_EM(phi_l)
    log_F = (

        # lens0_gamma ~ Normal(2.0, 0.2)
        norm.logpdf(
            gamma,
            loc=2.0,
            scale=0.2
        )

        # lens0_theta_E ~ Normal(theta_E_true, 0.01)
        + norm.logpdf(
            thetaE,
            loc=tp["lens0_theta_E"],
            scale=0.01
        )

        # lens0_e1 ~ Normal(0, sigma_e1)
        + norm.logpdf(
            e1,
            loc=0.0,
            scale=sigma_e1
        )

        # lens0_e2 ~ Normal(0, sigma_e2)
        + norm.logpdf(
            e2,
            loc=0.0,
            scale=sigma_e2
        )

        # lens1_gamma1 ~ Normal(gamma1_true, 0.01)
        + norm.logpdf(
            gamma1,
            loc=tp["lens1_gamma1"],
            scale=0.01
        )

        # lens1_gamma2 ~ Normal(gamma2_true, 0.01)
        + norm.logpdf(
            gamma2,
            loc=tp["lens1_gamma2"],
            scale=0.01
        )
    )

    return log_F


def compute_logZ_EM(ctx_em):
    """
    Laplace approximation to the EM evidence.
    """

    logp0_em = float(ctx_em["fisher"]["logp0"])
    H_em = np.asarray(ctx_em["fisher"]["H0"])

    k_em = H_em.shape[0]

    sign_em, logdet_em = np.linalg.slogdet(-H_em)

    if sign_em <= 0:
        raise ValueError(
            "Hessian does not have positive determinant after negation."
        )

    logZ_em = (
        logp0_em
        + 0.5 * k_em * np.log(2 * np.pi)
        - 0.5 * logdet_em
    )

    return logZ_em

## Step1: EM only evidence for the three

In [14]:
import numpyro.distributions as dist
from scipy.stats import gaussian_kde
from scipy.stats import norm
from scipy.special import logsumexp

In [15]:
em_results = {}

for idx, row in candidates.iterrows():

    print(
        f"\n================ EM system {idx} "
        f"({len(em_results)+1}/{Ncat}) ================"
    )

    #EM-only configuration
    cfg_em = row_to_cfg(row, sample_cfg,False)
    cfg_em["use_parameter_layout"] = True
    #Generate EM observation
    ctx_em = setup_em_observation(cfg=cfg_em)
    tp_em = ctx_em["truth_params"]

    #Set EM fiducial prior according to Otto's paper
    phi = row['deflector_pa']
    sigma_e = 2/((1+row['deflector_q'])**2) * 0.05
    sigma_e1 = abs(np.cos(2*phi)) * sigma_e
    sigma_e2 = abs(np.sin(2*phi)) * sigma_e

    ctx_em["cfg"]["priors"] = {
        'lens0_gamma': dist.Normal(2.0, 0.2),
          'lens0_theta_E': dist.Normal(tp_em['lens0_theta_E'], 0.01),
         #'lens0_e1': float(tp["lens0_e1"]), 
         #'lens0_e2': float(tp['lens0_e2']),
         'lens0_e1': dist.Normal(0, sigma_e1),
         'lens0_e2': dist.Normal(0, sigma_e2),
         'lens0_center_x': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_x']),
         'lens0_center_y': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_y']),
         #'lens1_gamma1': float(tp["lens1_gamma1"]),
         #'lens1_gamma2': float(tp["lens1_gamma2"]),
         'lens1_gamma1': dist.Normal(tp_em["lens1_gamma1"], 0.01),
         'lens1_gamma2': dist.Normal(tp_em["lens1_gamma2"], 0.01),
         'lens1_ra_0': 0.0,
         'lens1_dec_0': 0.0,
          'light0_R_sersic': float(tp_em["light0_R_sersic"]),
          'light0_n_sersic': float(tp_em["light0_n_sersic"]),
          'light0_amp': float(tp_em['light0_amp']),
         'light0_e1': float(tp_em["light0_e1"]),
         'light0_e2': float(tp_em["light0_e2"]),
         'light0_center_x': float(tp_em["light0_center_x"]),
         'light0_center_y': float(tp_em["light0_center_y"]),
          "noise_sigma_bkg": tp_em["noise_sigma_bkg"], 
        'source0_n_sersic': dist.Uniform(0.2, 10.0)
        } 

    
    # EM parameter estimation
    method = "deriv-approx"

    samples_em, truths_em = run_inference(
    ctx_em,
    mode= "EM-only", #"EM+GW",#"EM-only",#"GW-only",
    method=method,
    cfg={
        "output": {"json_tag": method},
        **({"inference": {"informed": True, "regularize": False,"num_chains": 8, "num_samples": 14500, "num_warmup": 9500}} if method == "deriv-approx" else {}),
        # "inference": {"num_chains": 12, "num_samples": 8000, "num_warmup": 8000, "n_fisher_samples": 10000},
        },
    )


    # EM Laplace evidence
    logZ_em = compute_logZ_EM(ctx_em)

    
    # KDE for p(phi_l | d_EM)
    samples_matrix_EM = np.vstack(
        [
            np.asarray(samples_em[p])
            for p in shared_params
        ]
    )

    kde_em = gaussian_kde(
        samples_matrix_EM
    )

    # Store everything needed later
    em_results[idx] = {
        "samples": samples_em,
        "truths": truths_em,
        "truth_params": tp_em,
        "logZ": logZ_em,
        "kde": kde_em,
        "row": row.copy()
    }

    print(
        f"Finished EM system {idx}: "
        f"log Z_EM = {logZ_em:.4f}"
    )


================ EM system 0 (1/3) ================
use_mst in probmodel: False
k_mst in keys_to_include: False
k_mst truth: None
keys_to_include: ['lens0_gamma', 'lens0_theta_E', 'lens0_e1', 'lens0_e2', 'lens1_gamma1', 'lens1_gamma2', 'source0_amp', 'source0_R_sersic', 'source0_n_sersic', 'source0_e1', 'source0_e2', 'source0_center_x', 'source0_center_y']
input_params: {'lens0_gamma': 2.095352206052913, 'lens0_theta_E': 1.492396410009919, 'lens0_e1': 0.009461834916696884, 'lens0_e2': 0.025757395488709124, 'lens1_gamma1': -0.0494144385765645, 'lens1_gamma2': -0.0287068717409659, 'source0_amp': 5.328224571538076, 'source0_R_sersic': 0.1639090601940684, 'source0_n_sersic': 1.0619585327394108, 'source0_e1': 0.025535188238353417, 'source0_e2': 0.7134448575695872, 'source0_center_x': -0.0611589253800559, 'source0_center_y': -0.0177206464670121}
Done with gradient
Done with Hessian
Scaled gradient at truth (g0 / sqrt|diag H0|):
  lens0_gamma               3.6603e-01
  lens0_theta_E         

/Users/sangchunyeh/workspace/research/internship/2026_CUHK/lens_reconstruction/src/gwemfish/simple_pipeline.py:2105: UserWarning: Pre-inference diagnostics failed for method='deriv-approx', mode='EM-only'. See the [diag] lines above. Set cfg['inference']['diagnostics'] = 'warn' to continue anyway, or 'off' to skip these checks.
  diagnostics_report = diagnose_system(


Hessian-informed MCMC:
  u0:            [ 2.09535221  1.49239641  0.00946183  0.0257574  -0.04941444 -0.02870687
  5.32822457  0.16390906  1.06195853  0.02553519  0.71344486 -0.06115893
 -0.01772065]
  Param sigmas:  [0.21087049 0.00432364 0.00699034 0.01868908 0.01122794 0.00703502
 0.68486671 0.03510663 0.17822842 0.03008335 0.02490104 0.01241475
 0.00465498]
  Scale:         1.0
  Perturb scale: 0.1
cov shape: (13, 13)


  0%|          | 0/24000 [00:00<?, ?it/s]

  0%|          | 0/24000 [00:00<?, ?it/s]

...

  0%|          | 0/24000 [00:00<?, ?it/s]

  0%|          | 0/24000 [00:00<?, ?it/s]

  0%|          | 0/24000 [00:00<?, ?it/s]

  0%|          | 0/24000 [00:00<?, ?it/s]



[lens0_e1]      	 max r_hat: 1.0000


[lens0_e2]      	 max r_hat: 1.0000


[lens0_gamma]   	 max r_hat: 1.0000


[lens0_theta_E]  	 max r_hat: 1.0000


[lens1_gamma1]  	 max r_hat: 1.0000


[lens1_gamma2]  	 max r_hat: 1.0000


[source0_R_sersic]  	 max r_hat: 1.0000


[source0_amp]   	 max r_hat: 1.0000


[source0_center_x]  	 max r_hat: 1.0000


[source0_center_y]  	 max r_hat: 1.0000


[source0_e1]    	 max r_hat: 1.0000


[source0_e2]    	 max r_hat: 1.0000


[source0_n_sersic]  	 max r_hat: 1.0000
Finished EM system 4: log Z_EM = 4750.6689


## Step 2: Run the GW evidence for only one target

In [16]:
gw_idx = 0
gw_row = candidates.iloc[0]

print(
    f"\n================ GW event = {gw_idx} ================"
)

cfg_gw = row_to_cfg(gw_row, sample_cfg, True)
cfg_gw["use_parameter_layout"] = True

#Generate EM and GW observation
ctx_post_em = setup_em_observation(cfg=cfg_gw)
ctx_gw = setup_gw_observation(ctx_post_em, cfg=cfg_gw)
tp_gw = ctx_gw["truth_params"]

#Set up GW fiducial prior according to Otto's paper
ctx_gw["cfg"]["priors"] = {
        "T_star": float(tp_gw["T_star"]),
        "dL": float(tp_gw["dL"]),
        'lens0_gamma': dist.Normal(2.0, 0.2),
          'lens0_theta_E': dist.Normal(tp_gw['lens0_theta_E'], 0.01),
         #'lens0_e1': float(tp["lens0_e1"]), 
         #'lens0_e2': float(tp['lens0_e2']),
         'lens0_e1': dist.Normal(tp_gw['lens0_e1'], sigma_e1),
         'lens0_e2': dist.Normal(tp_gw['lens0_e2'], sigma_e2),
         'lens0_center_x': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_x']),
         'lens0_center_y': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_y']),
         #'lens1_gamma1': float(tp["lens1_gamma1"]),
         #'lens1_gamma2': float(tp["lens1_gamma2"]),
         'lens1_gamma1': dist.Normal(tp_gw['lens1_gamma1'], 0.01),
         'lens1_gamma2': dist.Normal(tp_gw['lens1_gamma2'], 0.01),
         'lens1_ra_0': 0.0,
         'lens1_dec_0': 0.0,
          'light0_R_sersic': float(tp_gw["light0_R_sersic"]),
          'light0_n_sersic': float(tp_gw["light0_n_sersic"]),
          'light0_amp': float(tp_gw['light0_amp']),
         'light0_e1': float(tp_gw["light0_e1"]),
         'light0_e2': float(tp_gw["light0_e2"]),
         'light0_center_x': float(tp_gw["light0_center_x"]),
         'light0_center_y': float(tp_gw["light0_center_y"]),
          "noise_sigma_bkg": tp_gw["noise_sigma_bkg"],  
        'source0_n_sersic': dist.Uniform(0.2, 10.0)
        } 

#GW Parameter estimation
method = 'deriv-approx'
samples_gw, truths_gw = run_inference(
    ctx_gw,
    mode= "GW-only", 
    method=method,
    cfg={
        "output": {"json_tag": method},
        **({"inference": {"informed": True, "regularize": False,"num_chains": 8, "num_samples": 14500, "num_warmup": 9500}} if method == "deriv-approx" else {}),
        # "inference": {"num_chains": 12, "num_samples": 8000, "num_warmup": 8000, "n_fisher_samples": 10000},
    },
)

#Extract the GW marginal samples
samples_matrix_GW = np.vstack([np.asarray(samples_gw[p]) for p in shared_params])
n_gw_samples = samples_matrix_GW.shape[1]

#Compute the gw prior ratio
# p(Lambda | H_L) p(phi | H_L)
# --------------------------------
# F(Lambda, phi)
gw_prior_ratio = np.ones(n_gw_samples)



================ GW event = 0 ================
use_mst in probmodel: False
k_mst in keys_to_include: False
k_mst truth: None
keys_to_include: ['lens0_gamma', 'lens0_theta_E', 'lens0_e1', 'lens0_e2', 'lens1_gamma1', 'lens1_gamma2', 'image_x1', 'image_y1', 'image_x2', 'image_y2', 'image_x3', 'image_y3', 'image_x4', 'image_y4']
input_params: {'lens0_gamma': 2.095352206052913, 'lens0_theta_E': 1.492396410009919, 'lens0_e1': 0.009461834916696884, 'lens0_e2': 0.025757395488709124, 'lens1_gamma1': -0.0494144385765645, 'lens1_gamma2': -0.0287068717409659, 'image_x1': 0.10231557988085761, 'image_y1': -1.5879642059382244, 'image_x2': -0.9127052866363339, 'image_y2': 1.2874173099661597, 'image_x3': -1.3859810869840468, 'image_y3': -0.4676172940620972, 'image_x4': 1.2731751874071269, 'image_y4': 0.4738510777465847}
Done with gradient
Done with Hessian
Scaled gradient at truth (g0 / sqrt|diag H0|):
  lens0_gamma               3.0216e-02
  lens0_theta_E             2.5459e-03
  lens0_e1            

/Users/sangchunyeh/workspace/research/internship/2026_CUHK/lens_reconstruction/src/gwemfish/simple_pipeline.py:2105: UserWarning: Pre-inference diagnostics failed for method='deriv-approx', mode='GW-only'. See the [diag] lines above. Set cfg['inference']['diagnostics'] = 'warn' to continue anyway, or 'off' to skip these checks.
  diagnostics_report = diagnose_system(


Hessian-informed MCMC:
  u0:            [ 2.09535221  1.49239641  0.00946183  0.0257574  -0.04941444 -0.02870687
  0.10231558 -1.58796421 -0.91270529  1.28741731 -1.38598109 -0.46761729
  1.27317519  0.47385108]
  Param sigmas:  [0.17439344 0.0099845  0.01902789 0.01278214 0.00906406 0.00958742
 0.18156043 0.02190536 0.14443467 0.10330049 0.05710898 0.16372181
 0.06237299 0.14734935]
  Scale:         1.0
  Perturb scale: 0.1
cov shape: (14, 14)


  0%|          | 0/24000 [00:00<?, ?it/s]

  0%|          | 0/24000 [00:00<?, ?it/s]

...

  0%|          | 0/24000 [00:00<?, ?it/s]

  0%|          | 0/24000 [00:00<?, ?it/s]

  0%|          | 0/24000 [00:00<?, ?it/s]

  0%|          | 0/24000 [00:00<?, ?it/s]



[image_x1]      	 max r_hat: 1.0000


[image_x2]      	 max r_hat: 1.0000


[image_x3]      	 max r_hat: 1.0000


[image_x4]      	 max r_hat: 1.0000


[image_y1]      	 max r_hat: 1.0000


[image_y2]      	 max r_hat: 1.0000


[image_y3]      	 max r_hat: 1.0000


[image_y4]      	 max r_hat: 1.0000


[lens0_e1]      	 max r_hat: 1.0000


[lens0_e2]      	 max r_hat: 1.0000


[lens0_gamma]   	 max r_hat: 1.0000


[lens0_theta_E]  	 max r_hat: 1.0000


[lens1_gamma1]  	 max r_hat: 1.0000


[lens1_gamma2]  	 max r_hat: 1.0000


## Step 3: Evaluating Z_EM,i * p(phi | d_EM,i) / F_EM,i(phi) for each EM system on the GW samples

In [18]:
A_values = []

for idx, result in em_results.items():
    print(f"Evaluating EM catalogue contribution {idx}...")
    
    samples_em = result["samples"]
    kde_em = result["kde"]
    logZ_em = result["logZ"]
    tp_em = result["truth_params"]
    row_em = result["row"]

    # Evaluate p(phi | d_EM) at GW posterior samples
    p_em_at_gw = kde_em(samples_matrix_GW)

    # Prevent log(0)
    p_em_at_gw = np.maximum(p_em_at_gw, np.finfo(float).tiny)

    log_p_em_at_gw = np.log(p_em_at_gw)

    # Evaluate F_EM(phi) at the SAME GW samples
    gw_sample_dict = {p: np.asarray(samples_gw[p]) for p in shared_params}

    log_F_em_at_gw = log_F_EM(gw_sample_dict, tp_em, row_em)

    # EM likelihood factor:
    # Z_EM * p(phi | d_EM)
    # ----------------
    # F_EM(phi)

    log_integrand = (
        logZ_em
        + log_p_em_at_gw
        - log_F_em_at_gw
    )

    # Average over the marginal GW posterior
    log_A_i = logmeanexp(log_integrand)

    A_i = np.exp(log_A_i)
    A_values.append(A_i)

    print(f"    A_{idx} = {A_i:.6e}")

Evaluating EM catalogue contribution 0...


/var/folders/9s/6gn9tn517t98f_24r73rxxn40000gn/T/ipykernel_64768/1507262018.py:39: RuntimeWarning: overflow encountered in exp
  A_i = np.exp(log_A_i)


    A_0 = inf
Evaluating EM catalogue contribution 3...
    A_3 = inf
Evaluating EM catalogue contribution 4...
    A_4 = inf


In [20]:
catalog_EM_term = np.mean(A_values)

print("\nCatalogue EM term =", catalog_EM_term)


Catalogue EM term = inf


## Step 4: Add the incomplete-catalogue term

 numerator =
 < [ f/Ncat sum_i A_i(phi) + (1-f) ]
       w(phi,Lambda) >

 denominator = < w(phi,Lambda) >

In [21]:
numerator_samples = (
    f * catalog_EM_term
    + (1.0 - f)
) * gw_prior_ratio


numerator = np.mean(
    numerator_samples
)

denominator = np.mean(
    gw_prior_ratio
)

## Step 5 and FINAL STEP! Z_MM / Z_L

In [22]:
numerator_samples = (
    f * catalog_EM_term
    + (1.0 - f)
) * gw_prior_ratio


numerator = np.mean(
    numerator_samples
)

denominator = np.mean(
    gw_prior_ratio
)

In [23]:
gain = numerator / denominator

In [24]:
gain

np.float64(inf)